In [1]:
import os
from dotenv import load_dotenv

load_dotenv()


In [4]:
!pip install -q youtube-transcript-api langchain-community langchain-google-genai faiss-cpu tiktoken python-dotenv

In [5]:
from youtube_transcript_api import YouTubeTranscriptApi,TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings,ChatGoogleGenerativeAI
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate



# **Indexing (Doc Ingestion)**

In [11]:
!pip show youtube-transcript-api

Name: youtube-transcript-api
Version: 1.2.5
Summary: This is a python API which allows you to get the transcripts/subtitles for a given YouTube video. It also works for automatically generated subtitles, supports translating subtitles and it does not require a headless browser, like other selenium based solutions do!
Home-page: https://github.com/jdepoix/youtube-transcript-api
Author: Jonas Depoix
Author-email: jonas.depoix@web.de
License: MIT
Location: /usr/local/lib/python3.13/dist-packages
Requires: defusedxml, requests
Required-by: 


In [17]:
video_id = "6r0QlSamwR0"

try:
    api = YouTubeTranscriptApi()

    transcript = api.fetch(video_id, languages=['hi','en'])

    text = " ".join(
        chunk['text'] for chunk in transcript.to_raw_data()
    )

    print(text)

except TranscriptsDisabled:
    print("Transcripts are disabled for this video.")
except Exception as e:
    print("Error:", e)

सब्सक्राइब करने के बाद घंटा बजाएं ताकि आप दूसरा अपडेट घंटा भी मिस ना करें। दोस्ती इम्तिहान लेती है। [ __ ] की जान लेती है। हो गया तेरा हैप्पी दिवाली। ओ हैप्पी दिवाली उदाहर रही है। हां अजय। मैं बता रही हूं नाजिम इस बार मैं अनार बम फोड़ दूंगा। अनार बम फोड़ेगा। केला बम फोडूंगा मैं। साले केला केला बम फोड़ेगा तू मैं मुर्गा छाप फाड़ दूंगा पूरा कुत्ता छाप फाड़ दूंगा मैं सुतली बम फाड़ दूंगा नाड़ा बम फाड़ दूंगा मैं तेरी गांड ना फाड़ दूंगा अरे भोसड़ अरे ना असली मैं थोड़ी फाडूंगा मजाक कर रही हूं मैंने ये सुना है यार दैनिक भोसकर पे कि दिल्ली एनसीआर में अपने पटाखे बंद है मुझे तो फेक न्यूज़ लग रही है क्या दैनिक भोसड़ी भोसड़ी की अबे इसकी न्यूज़ कभी गलत नहीं होती है यार तो फिर टीवी पे देखो मैं बताया हूं नाजिम मैं बता रही हूं अगर ऐसा हुआ मैं तेरी गांड अरे अरे रुक जा ना फाड़ूंगा ऐसा नहीं होगा मैं तो मजाक कर रहा हूं तुझे लगता है सरकार ऐसा करेगी अबे जैन ऐसा हो ही नहीं सकता अगर हो गया ना ले चाबी तेरे हाथ में है ऐसा हो ही नहीं सकता ड्रैगिंग न्यूज़ दिल्ली एनसीआर में पटाखे बंद सुतली बम की तरह है एनसीआर वालों के लिए ये खबर ए

In [18]:
transcript

FetchedTranscript(snippets=[FetchedTranscriptSnippet(text='सब्सक्राइब करने के बाद घंटा बजाएं ताकि', start=0.24, duration=4.64), FetchedTranscriptSnippet(text='आप दूसरा अपडेट घंटा भी मिस ना करें।', start=2.56, duration=4.08), FetchedTranscriptSnippet(text='दोस्ती', start=4.88, duration=7.28), FetchedTranscriptSnippet(text='इम्तिहान लेती है। [\xa0__\xa0] की जान लेती', start=6.64, duration=7.119), FetchedTranscriptSnippet(text='है।', start=12.16, duration=4.32), FetchedTranscriptSnippet(text='हो गया तेरा हैप्पी दिवाली।', start=13.759, duration=5.201), FetchedTranscriptSnippet(text='ओ हैप्पी दिवाली उदाहर रही है।', start=16.48, duration=4.24), FetchedTranscriptSnippet(text='हां अजय।', start=18.96, duration=4.72), FetchedTranscriptSnippet(text='मैं बता रही हूं नाजिम इस बार मैं अनार बम', start=20.72, duration=5.76), FetchedTranscriptSnippet(text='फोड़ दूंगा। अनार बम फोड़ेगा। केला बम', start=23.68, duration=6.72), FetchedTranscriptSnippet(text='फोडूंगा मैं। साले केला केला बम फोड़ेगा तू', start=26

# **Text Splitting**

In [49]:
splitter=RecursiveCharacterTextSplitter(chunk_size=100,chunk_overlap=40)
chunks=splitter.create_documents([text])

In [50]:
len(chunks)

73

In [51]:
chunks[::-1]

[Document(metadata={}, page_content='सकता है? इसे सिर्फ मैं ही डिफ्यूज कर सकता हूं। [संगीत]'),
 Document(metadata={}, page_content='तुम्हें आरडीएक्स बम कितना खतरनाक होता है? पूरे जिले को उड़ा सकता है? इसे सिर्फ मैं ही डिफ्यूज कर'),
 Document(metadata={}, page_content='थू थू करिया तूने किसको लगा दिया? अबे सालो थू थू छोड़ो। पता है तुम्हें आरडीएक्स बम कितना खतरनाक होता'),
 Document(metadata={}, page_content='ट्रिपल2 ट्रिपल टू थू थू थू साले किसका नंबर दे दिया? साले मैं थू थू करिया तूने किसको लगा दिया? अबे'),
 Document(metadata={}, page_content='का नंबर दे मुझे फुलजड़ी का लिख ले चल दो दो दो टू टू टू टू ट्रिपल2 ट्रिपल टू थू थू थू साले किसका'),
 Document(metadata={}, page_content='बम है इसमें बम तूने किसने गोल लगा दी थी ना दे फुलजड़ी भाई का नंबर दे मुझे फुलजड़ी का लिख ले चल दो'),
 Document(metadata={}, page_content='दीजिए साले कमर में लगा दिया ना तो क्र में मिलेगा आरडीएक्स बम है इसमें बम तूने किसने गोल लगा दी थी'),
 Document(metadata={}, page_content='बम है आरडीएक्स है तो लगा लो आयोडक्स है तो क

# **Embedding Generation and Vector Storage**

In [52]:
embeddings=GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-2")
vector_store=FAISS.from_documents(chunks,embeddings)

In [53]:
vector_store.index_to_docstore_id

{0: 'cd47b435-6b50-4eac-b6c9-07a0dbd10567',
 1: 'f9b25375-c7da-4a61-8e2a-3427b5b79785',
 2: '121d811d-ca24-4abb-88e1-693e130e2db1',
 3: '3d1f6cfd-34e4-4556-b592-35e8d189fe58',
 4: 'edee57a8-2314-4736-bc87-4afc9a35444e',
 5: '079f4bdc-33e1-48b4-8bf3-412cfc6a9c65',
 6: 'ee7c7d4d-ddd7-4a92-8853-746fb5861ebc',
 7: 'c31257a3-f8d4-4206-8290-d940af01b790',
 8: '336ea05b-03aa-496d-a387-7bbb92db92b7',
 9: '698ccc90-40b0-49d7-a606-878317201f95',
 10: '99401954-0d5c-420a-a84f-ed934ec54c92',
 11: 'd272b6f2-c609-48b8-a6c7-8623c0f26e22',
 12: 'f193ca63-e821-41d3-863c-75c2cc0879db',
 13: 'cbcaea82-f2d9-4faa-95d7-5e9ef6bd87d5',
 14: '895469f2-4035-4606-b402-e5b68bde783f',
 15: '4c880888-19fa-4bc3-a0ba-a16404e51429',
 16: 'a1c9f1dd-6693-46aa-9e18-547187912fd4',
 17: 'dc1d1c5f-b500-4046-a1d3-da032bc51a34',
 18: '0f2ec66d-be02-457c-8251-c59efbc954fb',
 19: '7657d414-6a69-4820-9b23-48c21984b82f',
 20: '6114c922-a087-446e-86f8-99309ac12bbc',
 21: '57d9326e-af7f-4468-9538-9b0108303884',
 22: '9a33c555-4f09-

In [54]:
vector_store.get_by_ids(['d24e9f2b-988a-4eb8-89b8-fc5c441bff52'])

[]

# **Retrival**

In [55]:
retriever=vector_store.as_retriever(search_type="similarity",search_kwargs={"k":4})

In [56]:
retriever

VectorStoreRetriever(tags=['FAISS', 'GoogleGenerativeAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x7e151f98a850>, search_kwargs={'k': 4})

In [57]:
retriever.invoke('who is nazim')

[Document(id='d272b6f2-c609-48b8-a6c7-8623c0f26e22', metadata={}, page_content='पे देखो मैं बताया हूं नाजिम मैं बता रही हूं अगर ऐसा हुआ मैं तेरी गांड अरे अरे रुक जा ना फाड़ूंगा'),
 Document(id='2f22ef32-20ff-4df7-82b5-7474314f2d9f', metadata={}, page_content='का लिख ले। चल दो दो दो टू टू टू टू ट्रिपल टू ट्रिपल टू [संगीत] नाजिम क्या चाह रही है फिर सरकार?'),
 Document(id='d6e8fb81-719e-4b12-b3c2-f51619da1589', metadata={}, page_content='नाजिम क्या चाह रही है फिर सरकार? पटाखे ना फोड़े तो किसी के सर फोड़े। खोपड़े फोड़े। ओ फुलजड़ी भाई आ रहे'),
 Document(id='121d811d-ca24-4abb-88e1-693e130e2db1', metadata={}, page_content='है। हो गया तेरा हैप्पी दिवाली। ओ हैप्पी दिवाली उदाहर रही है। हां अजय। मैं बता रही हूं नाजिम इस बार')]

# **Augmentation**

In [58]:
llm=ChatGoogleGenerativeAI(model='gemini-3.5-flash', temperature=0.2)

In [59]:
prompt=PromptTemplate(
    template=""" you are the helpful assistance
    answer only from provided transcript context of vedio if context is insifficient just say you dont know.
    {context}
    question: {question}
    """,
    input_variables=['context','question']
)

In [85]:
question="what cities are they talking about and why "
retrived_docs=retriever.invoke(question)

In [86]:
retrived_docs

[Document(id='9366697a-9654-4123-9ef7-9336fcb2ef9d', metadata={}, page_content='में फोड़ेंगे अरे सरकार का नया एजेंडा है ये पब्लिसिटी कर रही है इको फ्रेंडली पटाखों की कौन सी सिटी'),
 Document(id='0f2ec66d-be02-457c-8251-c59efbc954fb', metadata={}, page_content='अपना एनसीआर में थोड़ी आवे है। अबे जाहिलों गुड़गांव भी एनसीआर में ही आता है। तू तू साले फिर टेंशन ले'),
 Document(id='1764c127-9153-4626-bdb1-3cf6ee6044b6', metadata={}, page_content='से डील कर रहे हो। ये क्या अलकायदा? अरे भैया फायदे की बात कर रहे थे। अरे हमारा तो फायदा ही ना हुआ।'),
 Document(id='a1c9f1dd-6693-46aa-9e18-547187912fd4', metadata={}, page_content='सुतली बम की तरह है एनसीआर वालों के लिए ये खबर एनसीआर में तो मैं भी रहूं तो तू क्यों टेंशन लेिया है')]

In [87]:
context_text='\n\n'.join(doc.page_content for doc in retrived_docs)

In [88]:
context_text

'में फोड़ेंगे अरे सरकार का नया एजेंडा है ये पब्लिसिटी कर रही है इको फ्रेंडली पटाखों की कौन सी सिटी\n\nअपना एनसीआर में थोड़ी आवे है। अबे जाहिलों गुड़गांव भी एनसीआर में ही आता है। तू तू साले फिर टेंशन ले\n\nसे डील कर रहे हो। ये क्या अलकायदा? अरे भैया फायदे की बात कर रहे थे। अरे हमारा तो फायदा ही ना हुआ।\n\nसुतली बम की तरह है एनसीआर वालों के लिए ये खबर एनसीआर में तो मैं भी रहूं तो तू क्यों टेंशन लेिया है'

In [89]:
final_prompt=prompt.invoke({'context':context_text,'question':question})

In [90]:
final_prompt

StringPromptValue(text=' you are the helpful assistance \n    answer only from provided transcript context of vedio if context is insifficient just say you dont know.\n    में फोड़ेंगे अरे सरकार का नया एजेंडा है ये पब्लिसिटी कर रही है इको फ्रेंडली पटाखों की कौन सी सिटी\n\nअपना एनसीआर में थोड़ी आवे है। अबे जाहिलों गुड़गांव भी एनसीआर में ही आता है। तू तू साले फिर टेंशन ले\n\nसे डील कर रहे हो। ये क्या अलकायदा? अरे भैया फायदे की बात कर रहे थे। अरे हमारा तो फायदा ही ना हुआ।\n\nसुतली बम की तरह है एनसीआर वालों के लिए ये खबर एनसीआर में तो मैं भी रहूं तो तू क्यों टेंशन लेिया है\n    question: what cities are they talking about and why \n    ')

# **Generation**

In [91]:
answer=llm.invoke(final_prompt)
print(answer.content)

[{'type': 'text', 'text': "Based on the provided transcript context:\n\n* **Cities/Regions mentioned:** **Gurgaon** and the **NCR** (National Capital Region).\n* **Why:** They are talking about them because of the government's new agenda/publicity regarding **eco-friendly firecrackers** (इको फ्रेंडली पटाखे) and discussing whether Gurgaon falls under the NCR region in relation to this news.", 'extras': {'signature': 'EqMZCqAZAWkUfROPXfrQvHx6HE4XVF0zkuYGQtU8OQQHdqgbEHLxMx1CfgJgm0RGy04htE/dfhnmWV/7g93Cg2CAE8Zd3unhSGpCUgeEHCQzTR7X0eRXz6/qCdZUiD4AXiC/2h/ZGYOJIy72C5aXajbRn3MqdBJXfGK0Z2LqO/eeov5kqsp5LGF3F+60EjSVTlOZrAq9AH3k/4VBmRMpAKCrgxYxzFAI5m6bY5UAuLIPZbGQItp3jNRBFOGz84IXGi5ofoDtpN/+zqfPk2dlT5rUNOPGRvp7dQ6GCofsAySPfGi/Uwp/Yoits7hehhynbD8ZdhF40LcdVM7c30Hy7JVpifPcq2URKrtg8oPoiIxSoFn6Lvwwg5o44/kbaQw/ieU0Oh7rc0KrCFQjM7qd2PSV1FR1EmTN6jjkKnPclr0nsgYGsO5OzsypCGV0yaSD45WJpaNbxFG+1OCnoUeLiNyKAqwcDQ3fyRckpRh5H56LaI2GPvtx/5LcEOIK6uc3ObXZwO8pry+fCEe/08UgsvjKNW9y5rsk4UmljmIZggZuyZcvGO1VIepF3VhF7YA6zb1z

In [92]:
question="summary and what gaaliyaan they use in the vedio "
retrived_docs=retriever.invoke(question)
context_text='\n\n'.join(doc.page_content for doc in retrived_docs)
final_prompt=prompt.invoke({'context':context_text,'question':question})
answer=llm.invoke(final_prompt)
print(answer.content)

[{'type': 'text', 'text': 'आधारित ट्रांसक्रिप्ट के अनुसार सारांश और उपयोग की गई गालियां नीचे दी गई हैं:\n\n### सारांश (Summary):\nट्रांसक्रिप्ट में कुछ लोगों के बीच बहस हो रही है जहाँ एक व्यक्ति पटाखों या बम की बात ("सुतली बम", "नाड़ा बम") को लेकर स्पष्टीकरण दे रहा है कि उसने यह बात सचमुच (असली में) नहीं कही थी। वह इस बात का खंडन कर रहा है कि उसका किसी आतंकवादी संगठन (जैसे अलकायदा) या पटाखों की डील से कोई संबंध है, और वह खुद पर लगे आरोपों को गलत बता रहा है।\n\n---\n\n### इस्तेमाल की गई गालियां (Profanities used in text):\n1. गांड\n2. भोसड़ / भोसड़ी की\n3. सालो', 'extras': {'signature': 'EtgeCtUeAWkUfRPqVRXhslFspLVIFDMZTrlp6ictDHzkJg6o4b9/uGhbn6oNb6PRaqv67EdkV1OBXCzlNCbKaXikOuWOTsreqgJu7/qvCSTRl+UVP2VNzm1+R3m7fVlA3QLhlkBs8FH4uJY/n/NCQ6vS8jpqnYsRMaIr0Y2zp4yZga6R/M59aGaJ6zqsqI3j4TYKXFzD09BW85dGmswuqaPFH7twCCZBJmRG8r9OWqpBKbSTEvjzKNfCD3Ctb3jHnYHI9uefYQ58gOEDlD6zMYz+4rwBjdv2qzTmgg/8O/uI51LMaPyba4KbAjO08Y5bYp43eGzi7kw1Wi1BKHNuf8h/Th9KxPK5uwYAXTPLHvsqsvKkGdv/zW7DZswHgINIvLHkEqXOnx4VtzUNOyhXuI

# **Building a Chain**


In [93]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser



In [94]:
def format_docs(retrieved_docs):
  context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
  return context_text

In [95]:
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),
    'question': RunnablePassthrough()
})

In [101]:
parallel_chain.invoke('summary of the story')

{'context': 'रहे थे। अरे हमारा तो फायदा ही ना हुआ। अभी तो डील ही ना हुई। तुम पहले आ गए और हमने करा क्या है\n\nअबे सरकार का एजेंडा होगा कोई। हमें तो पैसे मिलने हैं। पैसों से मतलब रख। ये बात भी सही है। अच्छा माल\n\nबहुत सारे। ठीक है। ना दे फुलजारी भाई का नंबर दे मुझे। फुलड़ी का लिख ले। चल दो दो दो टू टू टू टू\n\nका नंबर दे मुझे फुलजड़ी का लिख ले चल दो दो दो टू टू टू टू ट्रिपल2 ट्रिपल टू थू थू थू साले किसका',
 'question': 'summary of the story'}

In [102]:
parser = StrOutputParser()

In [103]:
main_chain = parallel_chain | prompt | llm | parser

In [105]:
main_chain.invoke('summary')

"**सारांश (Summary):**\n\nइस बातचीत में एक व्यक्ति 'फुलझड़ी' का फोन नंबर मांगता है, जिस पर दूसरा व्यक्ति '2' दोहराते हुए एक फर्जी या मजाकिया नंबर लिखवाता है। इसके बाद, सरकार के इरादों/एजेंडे के बारे में बात होती है, जहाँ एक व्यक्ति कहता है कि सरकार का जो भी एजेंडा हो, उन्हें सिर्फ अपने मिलने वाले पैसों से मतलब रखना चाहिए।"